In [3]:
from scipy.optimize import linprog

In [4]:
delay_probability=0.82
budget=20000
options={
    "Air Freight":{
        "cost":15000,
        "delay_days":3
    },
    "Secondary Supplier":{
        "cost":16500,
        "delay_days":5
    },
    "Delay Product Launch":{
        "cost":4000,
        "delay_days":14
    }
}

In [5]:
for action, values in options.items():
    score = values["cost"] + (values["delay_days"] * 1000)
    
    print(action, "→", score)

Air Freight → 18000
Secondary Supplier → 21500
Delay Product Launch → 18000


In [6]:
feasible_options = {
    action: values
    for action, values in options.items()
    if values["cost"] <= budget
}

feasible_options

{'Air Freight': {'cost': 15000, 'delay_days': 3},
 'Secondary Supplier': {'cost': 16500, 'delay_days': 5},
 'Delay Product Launch': {'cost': 4000, 'delay_days': 14}}

In [7]:
best_action = min(
    feasible_options,
    key=lambda action:
        feasible_options[action]["cost"]
        + feasible_options[action]["delay_days"] * 1000
)

best_action

'Air Freight'

In [8]:
import numpy as np
import pandas as pd

In [9]:
shipment={
    "shipment_id": "SHP01024",
    "order_quantity": 1200,
    "inventory_level": 800,
    "supplier_capacity": 5000,
    "shipping_cost": 12000,
    "delay_probability": 0.82
}

In [10]:
shipment

{'shipment_id': 'SHP01024',
 'order_quantity': 1200,
 'inventory_level': 800,
 'supplier_capacity': 5000,
 'shipping_cost': 12000,
 'delay_probability': 0.82}

In [11]:
actions=pd.DataFrame({
    "action":[
        "Air Freight",
        "Secondary supplier",
        "Delay Product Launch"
    ],
    "cost":[
        15000,
        16500,
        4000
    ],
    "delay_days":[
        3,5,14
    ],
    "capacity":[
        2000,
        3000,
        1200
    ]
})

In [12]:
actions

,action,cost,delay_days,capacity
0,Air Freight,15000,3,2000
1,Secondary supplier,16500,5,3000
2,Delay Product Launch,4000,14,1200


In [13]:
delay_penalty=1000
actions["objective_cost"]=(
    actions["cost"]+
    actions["delay_days"]*delay_penalty
)

In [14]:
actions

,action,cost,delay_days,capacity,objective_cost
0,Air Freight,15000,3,2000,18000
1,Secondary supplier,16500,5,3000,21500
2,Delay Product Launch,4000,14,1200,18000


In [15]:
c=actions["objective_cost"].values

In [16]:
c

array([18000, 21500, 18000])

In [17]:
A_eq = np.array([
    [1, 1, 1]
])

b_eq = np.array([1])

In [18]:
budget=20000

In [19]:
A_ub = np.array([
    actions["cost"].values
])

b_ub = np.array([
    budget
])

In [20]:
bounds = [
    (0, 1),
    (0, 1),
    (0, 1)
]

In [21]:
result = linprog(
    c=c,
    A_ub=A_ub,
    b_ub=b_ub,
    A_eq=A_eq,
    b_eq=b_eq,
    bounds=bounds,
    method="highs"
)

In [22]:
result.success

True

In [23]:
result.x

array([1., 0., 0.])

In [24]:
selected_index = np.argmax(result.x)

selected_action = actions.iloc[selected_index]["action"]

selected_action

'Air Freight'

In [25]:
print("Recommended Action:", selected_action)
print(
    "Estimated Cost: $",
    actions.iloc[selected_index]["cost"]
)
print(
    "Expected Delay:",
    actions.iloc[selected_index]["delay_days"],
    "days"
)

Recommended Action: Air Freight
Estimated Cost: $ 15000
Expected Delay: 3 days


In [26]:
shipment = {
    "order_quantity": 1200,
    "inventory_level": 800,
    "supplier_capacity": 5000,
    "delay_probability": 0.82
}

budget = 20000
max_acceptable_delay = 7

In [27]:
actions = pd.DataFrame({
    "action": [
        "Air Freight",
        "Secondary Supplier",
        "Delay Product Launch"
    ],
    
    "cost": [
        15000,
        16500,
        4000
    ],
    
    "delay_days": [
        3,
        5,
        14
    ],
    
    "capacity": [
        2000,
        3000,
        1200
    ],
    
    "risk_reduction": [
        0.75,
        0.55,
        0.10
    ]
})

actions

,action,cost,delay_days,capacity,risk_reduction
0,Air Freight,15000,3,2000,0.75
1,Secondary Supplier,16500,5,3000,0.55
2,Delay Product Launch,4000,14,1200,0.10


In [28]:
actions["remaining_risk"] = (
    shipment["delay_probability"]
    * (1 - actions["risk_reduction"])
)

actions

,action,cost,delay_days,capacity,risk_reduction,remaining_risk
0,Air Freight,15000,3,2000,0.75,0.205
1,Secondary Supplier,16500,5,3000,0.55,0.369
2,Delay Product Launch,4000,14,1200,0.10,0.738


In [29]:
actions["budget_feasible"] = (
    actions["cost"] <= budget
)

In [30]:
actions["delay_feasible"] = (
    actions["delay_days"] <= max_acceptable_delay
)

In [31]:
actions["capacity_feasible"] = (
    actions["capacity"] >= shipment["order_quantity"]
)

In [32]:
actions[
    [
        "action",
        "cost",
        "delay_days",
        "remaining_risk",
        "budget_feasible",
        "delay_feasible",
        "capacity_feasible"
    ]
]

,action,cost,delay_days,remaining_risk,budget_feasible,delay_feasible,capacity_feasible
0,Air Freight,15000,3,0.205,True,True,True
1,Secondary Supplier,16500,5,0.369,True,True,True
2,Delay Product Launch,4000,14,0.738,True,False,True


In [33]:
feasible_actions = actions[
    actions["budget_feasible"]
    & actions["delay_feasible"]
    & actions["capacity_feasible"]
].copy()

feasible_actions

,action,cost,delay_days,capacity,risk_reduction,remaining_risk,budget_feasible,delay_feasible,capacity_feasible
0,Air Freight,15000,3,2000,0.75,0.205,True,True,True
1,Secondary Supplier,16500,5,3000,0.55,0.369,True,True,True


In [34]:
cost_weight = 0.4
risk_weight = 0.6

feasible_actions["score"] = (
    cost_weight
    * (feasible_actions["cost"] / budget)
    +
    risk_weight
    * feasible_actions["remaining_risk"]
)

In [35]:
recommendations = feasible_actions.sort_values(
    "score"
).reset_index(drop=True)

recommendations[
    [
        "action",
        "cost",
        "delay_days",
        "remaining_risk",
        "score"
    ]
]

,action,cost,delay_days,remaining_risk,score
0,Air Freight,15000,3,0.205,0.4230
1,Secondary Supplier,16500,5,0.369,0.5514


In [36]:
best_action = recommendations.iloc[0]

print("Recommended Action:", best_action["action"])
print("Cost: $", best_action["cost"])
print("Expected Delay:", best_action["delay_days"], "days")
print(
    "Remaining Delay Risk:",
    round(best_action["remaining_risk"] * 100, 2),
    "%"
)

Recommended Action: Air Freight
Cost: $ 15000
Expected Delay: 3 days
Remaining Delay Risk: 20.5 %


In [37]:
recommendations[
    ["action", "cost", "delay_days", "remaining_risk", "score"]
]

,action,cost,delay_days,remaining_risk,score
0,Air Freight,15000,3,0.205,0.4230
1,Secondary Supplier,16500,5,0.369,0.5514
